
# Student–University Networks — Enhanced Version 2

This notebook incorporates the additional refinements:

- **Level categories**
  - `Doctor`, `Doctorate`, and `Licentiate` are merged into `Doctorate`.
  - Missing levels are represented as `Unspecified`.
  - Level controls use **multiple selection**.
- **University labels**
  - `University_clean` remains the internal network key.
  - Visible university labels use the most frequent raw `University` value corresponding to each cleaned key.
- **Projected network**
  - Louvain communities can optionally color nodes.
  - Communities are filtered by a **community-size interval** rather than community ID.
- **Spatial network**
  - Same raw university display labels and community-size interval filtering.
- **Sankeys**
  - Source/start and target/end levels are filtered independently.
  - The multi-stage Sankey has separate Stage 1 / Stage 2 / Stage 3 level filters.
  - The multi-stage Sankey now uses three consecutive observed education stages across **all countries**, so institutions outside China and the U.S. are included naturally.


In [ ]:

from pathlib import Path
from collections import defaultdict, Counter
import itertools
import json
import html

import numpy as np
import pandas as pd
import networkx as nx

from IPython.display import display, IFrame

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

DATA_PATH = Path("educ_geoloc.csv")
if not DATA_PATH.exists():
    alt = Path("data/educ_geoloc.csv")
    if alt.exists():
        DATA_PATH = alt

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head())


In [ ]:

# ---------------------------
# Cleaning and normalization
# ---------------------------

df = df_raw[
    df_raw["NameID"].notna() &
    df_raw["University_clean"].notna()
].copy()

def clean_string(v):
    return "" if pd.isna(v) else str(v).strip()

def normalize_level(v):
    if pd.isna(v) or not str(v).strip():
        return "Unspecified"

    level = str(v).strip()

    if level.lower() in {"doctor", "doctorate", "licentiate"}:
        return "Doctorate"
    
    if level.lower() in {"diploma", "certificate"}:
            return "Other degree"

    return level

df["_student"] = df["NameID"].astype(str).str.strip()
df["_student_name"] = (
    df["FullName"]
    .fillna(df["FullName_PY"])
    .fillna("")
    .astype(str)
    .str.strip()
)
df["_uni"] = df["University_clean"].astype(str).str.strip()
df["_year_num"] = pd.to_numeric(df["Year"], errors="coerce")
df["_level_cat"] = df["Level"].apply(normalize_level)
df["_discipline"] = df["discipline"].fillna("").astype(str).str.strip()

YEAR_MIN = int(df["_year_num"].min())
YEAR_MAX = int(df["_year_num"].max())

LEVEL_CATEGORIES = sorted(
    df["_level_cat"].unique().tolist(),
    key=lambda x: (x == "Unspecified", x)
)

UNIVERSITY_KEYS = sorted(df["_uni"].unique().tolist())

print("Normalized levels:")
display(df["_level_cat"].value_counts().rename("records").to_frame())


In [ ]:

# ---------------------------
# Broad discipline-family taxonomy ("Field")
# ---------------------------

FIELD_MAP = {
    # Humanities
    "Drama": "Humanities",
    "English": "Humanities",
    "Fine Arts": "Humanities",
    "History": "Humanities",
    "Language": "Humanities",
    "Literature": "Humanities",
    "Music": "Humanities",
    "Philosophy": "Humanities",
    "Photography": "Humanities",
    "Theology": "Humanities",
    "Library Science": "Humanities",

    # Social Sciences
    "Anthropology": "Social Sciences",
    "Economics": "Social Sciences",
    "Foreign Service": "Social Sciences",
    "Geography": "Social Sciences",
    "Journalism": "Social Sciences",
    "Law": "Social Sciences",
    "Political Science": "Social Sciences",
    "Psychology": "Social Sciences",
    "Social Science": "Social Sciences",
    "Social Welfare": "Social Sciences",
    "Sociology": "Social Sciences",
    "Home Economics": "Social Sciences",
    
    # Education
    "Education": "Education",
    "Physical Education": "Education",
    "Child Welfare": "Education",

    # Business & Administration
    "Accounting": "Business & Administration",
    "Banking & Finance": "Business & Administration",
    "Business": "Business & Administration",
    "Industrial Management": "Business & Administration",
    "Public Administration": "Business & Administration",
    "Railroad Administration": "Business & Administration",
    "Transportation": "Business & Administration",
    "Architecture": "Business & Administration",
    "City Planning": "Business & Administration",

    # Engineering
    "Aeronautical Engineering": "Engineering",
    "Automotive Engineering": "Engineering",
    "Chemical Engineering": "Engineering",
    "Civil Engineering": "Engineering",
    "Electrical Engineering": "Engineering",
    "Engineering": "Engineering",
    "Hydroelectric Engineering": "Engineering",
    "Marine Enginerring": "Engineering",
    "Mechanical Engineering": "Engineering",
    "Metallurgy": "Engineering",
    "Mining Engineering": "Engineering",
    "Radio Engineering": "Engineering",
    "Textile Engineering": "Engineering",

    # Physical Sciences
    "Chemistry": "Physical Sciences",
    "Mathematics": "Physical Sciences",

    # Biological Sciences
    "Bacteriology": "Biological Sciences",
    "Biochemistry": "Biological Sciences",
    "Biology": "Biological Sciences",
    "Botany": "Biological Sciences",
    "Entomology": "Biological Sciences",
    "Parasitology": "Biological Sciences",
    "Physiology": "Biological Sciences",
    "Plant Pathology": "Biological Sciences",
    "Zoology": "Biological Sciences",

    # Earth & Environmental Sciences
    "Climateology": "Earth & Environmental Sciences",
    "Forestry": "Earth & Environmental Sciences",
    "Geology": "Earth & Environmental Sciences",
    "Meteorology": "Earth & Environmental Sciences",
    "Soil Science": "Earth & Environmental Sciences",

    # Agricultural Sciences
    "Agricultural Economics": "Agricultural Sciences",
    "Agriculture": "Agricultural Sciences",
    "Agrononmy": "Agricultural Sciences",
    "Animal Husbandry": "Agricultural Sciences",
    "Horticulture": "Agricultural Sciences",

    # Health Sciences
    "Dentistry": "Health Sciences",
    "Dietetics": "Health Sciences",
    "Hygiene": "Health Sciences",
    "Medicine": "Health Sciences",
    "Neurology": "Health Sciences",
    "Nursing": "Health Sciences",
    "Nutrition": "Health Sciences",
    "Obstetrics": "Health Sciences",
    "Pharmacy": "Health Sciences",
    "Psychiatry": "Health Sciences",
    "Public Health": "Health Sciences",
    "Surgery": "Health Sciences",
    "Veterinary Medicine": "Health Sciences",

}

def discipline_to_field(discipline):
    d = clean_string(discipline)
    if not d:
        return "Unspecified"
    return FIELD_MAP.get(d, "Other / Interdisciplinary")

df["_field"] = df["discipline"].apply(discipline_to_field)

FIELD_CATEGORIES = sorted(
    df["_field"].unique().tolist(),
    key=lambda x: (x == "Unspecified", x)
)

print("Field categories:")
display(df["_field"].value_counts().rename("records").to_frame())



## Field taxonomy

The broad `Field` variable is an ad hoc analytical grouping derived from the original `discipline` values. It is intended for exploratory filtering rather than as an official disciplinary classification. The mapping is explicit in the notebook so it can be reviewed and edited.

Current families are:

- Humanities
- Social Sciences
- Business & Administration
- Engineering
- Physical Sciences
- Biological Sciences
- Earth & Environmental Sciences
- Agricultural Sciences
- Health Sciences
- Architecture & Planning
- Education
- Other / Interdisciplinary
- Unspecified


In [ ]:

# ---------------------------
# University display labels
# ---------------------------
# Internal graph key = University_clean
# Visible label = modal (most frequent) raw University value for that cleaned key.

university_display = {}

for clean_name, g in df.groupby("_uni"):
    raw_labels = [
        clean_string(x)
        for x in g["University"].dropna().tolist()
        if clean_string(x)
    ]

    university_display[clean_name] = (
        Counter(raw_labels).most_common(1)[0][0]
        if raw_labels else clean_name
    )

# Useful options sorted by visible label.
UNIVERSITY_OPTIONS = sorted(
    UNIVERSITY_KEYS,
    key=lambda u: university_display[u].lower()
)

display_map_df = pd.DataFrame({
    "University_clean": UNIVERSITY_OPTIONS,
    "University_display": [university_display[u] for u in UNIVERSITY_OPTIONS],
})

display(display_map_df.head(20))


In [ ]:

# ---------------------------
# Metadata helpers
# ---------------------------

student_meta = {}
for sid, g in df.groupby("_student"):
    names = [x for x in g["_student_name"].tolist() if x]
    student_meta[sid] = {
        "id": "s::" + sid,
        "key": sid,
        "label": names[0] if names else sid,
        "name": names[0] if names else "",
        "type": "student",
    }

uni_meta = {}
for uni, g in df.groupby("_uni"):
    countries = [clean_string(x) for x in g["Country"].dropna().tolist() if clean_string(x)]
    cities = [clean_string(x) for x in g["City"].dropna().tolist() if clean_string(x)]
    states = [clean_string(x) for x in g["Province_State"].dropna().tolist() if clean_string(x)]

    uni_meta[uni] = {
        "id": "u::" + uni,
        "key": uni,
        "label": university_display[uni],
        "clean_label": uni,
        "country": Counter(countries).most_common(1)[0][0] if countries else "",
        "city": Counter(cities).most_common(1)[0][0] if cities else "",
        "state": Counter(states).most_common(1)[0][0] if states else "",
        "type": "university",
    }

def normalize_country(country):
    c = (country or "").strip().lower()

    if c in {
        "china", "people's republic of china",
        "pr china", "p.r. china", "republic of china"
    }:
        return "China"

    if c in {
        "united states", "united states of america",
        "usa", "u.s.", "us", "u.s.a."
    }:
        return "US"

    return "Other"

university_country_class = {
    u: normalize_country(meta["country"])
    for u, meta in uni_meta.items()
}


In [ ]:

# ---------------------------
# HTML helpers
# ---------------------------

def json_compact(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

def option_tags(values, labels=None):
    if labels is None:
        labels = {v: v for v in values}

    return "".join(
        f'<option value="{html.escape(str(v), quote=True)}">'
        f'{html.escape(str(labels.get(v, v)))}'
        f'</option>'
        for v in values
    )

def multi_level_select(select_id, label, size=6):
    return f"""
    <div class="control">
      <label>{html.escape(label)}</label>
      <select id="{select_id}" multiple size="{size}">
        {option_tags(LEVEL_CATEGORIES)}
      </select>
      <span class="small">No selection = all levels</span>
    </div>
    """

common_head = """
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<style>
body{font-family:Inter,system-ui,Arial,sans-serif;margin:0;background:#f6f7f9;color:#17202a}
header{padding:14px 18px;background:white;border-bottom:1px solid #ddd}
h1{font-size:19px;margin:0 0 5px}.sub{font-size:12px;color:#667085}
.controls{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;background:white;border-bottom:1px solid #ddd;align-items:end}
.control{display:flex;flex-direction:column;gap:3px;min-width:110px}.control label{font-size:11px;font-weight:650;color:#475467}
input,select,button{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}
select[multiple]{min-width:150px;height:38px;min-height:38px;padding:3px;vertical-align:bottom}
button{cursor:pointer}.stat{margin-left:auto;font-size:12px;color:#475467;padding:8px}
#net,#map{height:calc(100vh - 180px);min-height:580px;background:white}
.small{font-size:11px;color:#667085}.active-btn{font-weight:700;border-color:#667085}
.inline-check{display:flex;align-items:center;gap:6px;min-height:36px}
</style>
"""


## 1. Bipartite network — multi-level selection

In [ ]:

bipartite_nodes = list(student_meta.values()) + list(uni_meta.values())

bipartite_edges = []

for i, r in df.reset_index(drop=True).iterrows():
    bipartite_edges.append({
        "id": f"e{i}",
        "from": "s::" + r["_student"],
        "to": "u::" + r["_uni"],
        "year": None if pd.isna(r["_year_num"]) else int(r["_year_num"]),
        "level": r["_level_cat"],
        "field": r["_field"],
        "discipline": r["_discipline"],
        "title": (
            f"{r['_student_name'] or r['_student']} → {university_display[r['_uni']]}"
            f"<br>Year: {'' if pd.isna(r['_year_num']) else int(r['_year_num'])}"
            f"<br>Level: {r['_level_cat']}"
            f"<br>Discipline: {r['_discipline'] or 'n/a'}"
        ),
    })

print("Bipartite nodes:", len(bipartite_nodes))
print("Bipartite edges:", len(bipartite_edges))


In [ ]:

bip_html = f"""<!doctype html>
<html>
<head>
<title>Bipartite student–university network</title>
{common_head}
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
</head>
<body>

<header>
<h1>Student–University Bipartite Network</h1>
<div class="sub">
Level categories are normalized; select multiple levels with Ctrl/Cmd-click. Node size is proportional to visible degree.
</div>
</header>

<div class="controls">

<div class="control">
<label>Student name / ID</label>
<input id="studentSearch" placeholder="type name or ID">
</div>

<div class="control">
<label>University</label>
<select id="uniFilter">
<option value="">All</option>
{option_tags(UNIVERSITY_OPTIONS, university_display)}
</select>
</div>

<div class="control">
<label>Year from</label>
<input id="yearMin" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MIN}">
</div>

<div class="control">
<label>Year to</label>
<input id="yearMax" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MAX}">
</div>

{multi_level_select("levelFilter", "Level(s)", size=6)}

<div class="control">
<label>Field</label>
<select id="fieldFilter">
<option value="">All</option>
{option_tags(FIELD_CATEGORIES)}
</select>
</div>

<div class="control">
<label>Discipline</label>
<select id="discFilter">
<option value="">All</option>
{option_tags(sorted([x for x in df["_discipline"].unique() if x]))}
</select>
</div>

<div class="control">
<label>Unknown year</label>
<select id="unknownYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Physics</label>
<div>
<button type="button" id="stopBtn">Stop</button>
<button type="button" id="slowBtn">Slow</button>
<button type="button" id="normalBtn" class="active-btn">Normal</button>
<button type="button" id="fastBtn">Fast</button>
</div>
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="net"></div>

<script>
const ALL_NODES={json_compact(bipartite_nodes)};
const ALL_EDGES={json_compact(bipartite_edges)};
const container=document.getElementById("net");
let network=null;
let physicsMode="normal";

function makeTooltip(lines){{
    const div=document.createElement("div");

    lines.forEach(line=>{{
        const row=document.createElement("div");

        if(line.bold){{
            const strong=document.createElement("strong");
            strong.textContent=line.text;
            row.appendChild(strong);
        }} else {{
            row.textContent=line.text;
        }}

        div.appendChild(row);
    }});

    return div;
}}

function selectedValues(el){{
    return [...el.selectedOptions].map(o=>o.value);
}}

function sizeScale(v,minV,maxV,a,b){{
    if(maxV<=minV) return (a+b)/2;
    return a+(Math.sqrt(v)-Math.sqrt(minV))*(b-a)/(Math.sqrt(maxV)-Math.sqrt(minV));
}}

function physicsOptions(mode){{
    if(mode==="stop") return {{enabled:false}};

    const speed={{
        slow:{{timestep:.22,maxVelocity:18}},
        normal:{{timestep:.50,maxVelocity:38}},
        fast:{{timestep:.85,maxVelocity:70}}
    }}[mode];

    return {{
        enabled:true,
        solver:"barnesHut",
        timestep:speed.timestep,
        maxVelocity:speed.maxVelocity,
        barnesHut:{{gravitationalConstant:-6000,springLength:95}},
        stabilization:{{enabled:false}}
    }};
}}

function setPhysics(mode){{
    physicsMode=mode;
    ["stopBtn","slowBtn","normalBtn","fastBtn"]
        .forEach(id=>document.getElementById(id).classList.remove("active-btn"));

    document.getElementById(
        mode==="stop"?"stopBtn":
        mode==="slow"?"slowBtn":
        mode==="fast"?"fastBtn":"normalBtn"
    ).classList.add("active-btn");

    if(network) network.setOptions({{physics:physicsOptions(mode)}});
}}

function render(){{
    let y1=parseInt(yearMin.value,10);
    let y2=parseInt(yearMax.value,10);
    if(y1>y2) [y1,y2]=[y2,y1];

    const q=studentSearch.value.trim().toLowerCase();
    const uf=uniFilter.value;
    const levels=selectedValues(levelFilter);
    const field=fieldFilter.value;
    const discipline=discFilter.value;
    const includeUnknown=unknownYear.value==="include";

    let edges=ALL_EDGES.filter(e=>{{
        const yearOK=e.year===null ? includeUnknown : (e.year>=y1 && e.year<=y2);
        const levelOK=levels.length===0 || levels.includes(e.level);

        return yearOK &&
            levelOK &&
            (!field || e.field===field) &&
            (!discipline || e.discipline===discipline) &&
            (!uf || e.to==="u::"+uf);
    }});

    if(q){{
        const matches=new Set(
            ALL_NODES.filter(n=>
                n.type==="student" &&
                (
                    (n.name||"").toLowerCase().includes(q) ||
                    (n.key||"").toLowerCase().includes(q)
                )
            ).map(n=>n.id)
        );
        edges=edges.filter(e=>matches.has(e.from));
    }}

    const ids=new Set();
    const neighbors={{}};

    edges.forEach(e=>{{
        ids.add(e.from); ids.add(e.to);
        if(!neighbors[e.from]) neighbors[e.from]=new Set();
        if(!neighbors[e.to]) neighbors[e.to]=new Set();
        neighbors[e.from].add(e.to);
        neighbors[e.to].add(e.from);
    }});

    const degVals=[...ids].map(id=>neighbors[id]?.size||1);
    const minD=Math.min(...degVals,1);
    const maxD=Math.max(...degVals,1);

    const nodes=ALL_NODES.filter(n=>ids.has(n.id)).map(n=>{{
        const d=neighbors[n.id]?.size||1;
        const size=sizeScale(
            d,minD,maxD,
            n.type==="student"?7:10,
            n.type==="student"?24:38
        );

        if(n.type==="student"){{
            return {{
                ...n,
                shape:"dot",
                color:"#4c78a8",
                size,
                title:makeTooltip([
                    {{text:n.name||n.key,bold:true}},
                    {{text:`ID: ${{n.key}}`}},
                    {{text:`Visible university degree: ${{d}}`}}
                ])
            }};
        }}

        const location=[n.city,n.state]
            .filter(x=>x && x.trim())
            .join(", ");

        return {{
            ...n,
            shape:"diamond",
            color:"#f28e2b",
            size,
            title:makeTooltip([
                {{text:n.label,bold:true}},
                ...(location ? [{{text:location}}] : []),
                ...(n.country ? [{{text:n.country}}] : []),
                {{text:`Visible student degree: ${{d}}`}}
            ])
        }};
    }});

    const visualEdges=edges.map(e=>{{
        const student=ALL_NODES.find(n=>n.id===e.from);
        const university=ALL_NODES.find(n=>n.id===e.to);

        return {{
            ...e,
            color:{{color:"#b8c0cc"}},
            width:1,
            smooth:{{type:"dynamic"}},
            title:makeTooltip([
                {{
                    text:`${{student?.label || student?.key || e.from}} → ${{university?.label || e.to}}`,
                    bold:true
                }},
                ...(e.year!==null ? [{{text:`Year: ${{e.year}}`}}] : []),
                {{text:`Level: ${{e.level || "Unspecified"}}`}},
                ...(e.field ? [{{text:`Field: ${{e.field}}`}}] : []),
                ...(e.discipline ? [{{text:`Discipline: ${{e.discipline}}`}}] : [])
            ])
        }};
    }});

    if(network) network.destroy();

    network=new vis.Network(
        container,
        {{nodes:new vis.DataSet(nodes),edges:new vis.DataSet(visualEdges)}},
        {{
            physics:physicsOptions(physicsMode),
            interaction:{{hover:true,navigationButtons:true}},
            nodes:{{font:{{size:11}}}}
        }}
    );

    stat.textContent=`${{nodes.length}} nodes · ${{edges.length}} edges · years ${{y1}}–${{y2}}`;
}}

["studentSearch","uniFilter","yearMin","yearMax","levelFilter","fieldFilter","discFilter","unknownYear"]
.forEach(id=>document.getElementById(id).addEventListener(id==="studentSearch"?"input":"change",render));

stopBtn.onclick=()=>setPhysics("stop");
slowBtn.onclick=()=>setPhysics("slow");
normalBtn.onclick=()=>setPhysics("normal");
fastBtn.onclick=()=>setPhysics("fast");

reset.onclick=()=>{{
    studentSearch.value="";
    uniFilter.value="";
    yearMin.value="{YEAR_MIN}";
    yearMax.value="{YEAR_MAX}";
    [...levelFilter.options].forEach(o=>o.selected=false);
    fieldFilter.value="";
    discFilter.value="";
    unknownYear.value="include";
    physicsMode="normal";
    render();
    setPhysics("normal");
}};

render();
</script>
</body>
</html>
"""

bip_path = OUTPUT_DIR / "01_bipartite.html"
bip_path.write_text(bip_html, encoding="utf-8")
print("Saved:", bip_path.resolve())


## 2. Projected university network with community-size filtering

In [ ]:

pair_students = defaultdict(set)
uni_students = defaultdict(set)

for sid, g in df.groupby("_student"):
    universities = sorted(set(g["_uni"]))

    for u in universities:
        uni_students[u].add(sid)

    for a, b in itertools.combinations(universities, 2):
        pair_students[(a,b)].add(sid)

G = nx.Graph()
G.add_nodes_from(UNIVERSITY_KEYS)

for (a,b), students in pair_students.items():
    G.add_edge(a,b,weight=len(students))

degree_centrality = nx.degree_centrality(G)
weighted_degree = dict(G.degree(weight="weight"))
betweenness_centrality = nx.betweenness_centrality(G,weight=None,normalized=True)
closeness_centrality = nx.closeness_centrality(G)
pagerank = nx.pagerank(G,weight="weight")
edge_betweenness = nx.edge_betweenness_centrality(G,weight=None,normalized=True)

if hasattr(nx.community,"louvain_communities"):
    communities = nx.community.louvain_communities(G,weight="weight",seed=42)
    community_method = "Louvain"
else:
    communities = list(nx.community.greedy_modularity_communities(G,weight="weight"))
    community_method = "Greedy modularity"

communities = sorted(communities,key=len,reverse=True)
community_of = {}
community_sizes = {}

for cid,members in enumerate(communities,start=1):
    community_sizes[cid]=len(members)
    for u in members:
        community_of[u]=cid

MAX_COMMUNITY_SIZE = max(community_sizes.values())

projection_nodes = []

for u in G.nodes():
    cid=community_of.get(u,0)

    projection_nodes.append({
        "id":u,
        "label":university_display[u],
        "clean_label":u,
        "students":len(uni_students[u]),
        "degree":G.degree(u),
        "degree_centrality":degree_centrality.get(u,0),
        "weighted_degree":weighted_degree.get(u,0),
        "betweenness":betweenness_centrality.get(u,0),
        "closeness":closeness_centrality.get(u,0),
        "pagerank":pagerank.get(u,0),
        "community":cid,
        "community_size":community_sizes.get(cid,1),
        "country":uni_meta[u]["country"],
        "city":uni_meta[u]["city"],
        "state":uni_meta[u]["state"],
    })

projection_edges = []

for j,(a,b,d) in enumerate(G.edges(data=True)):
    projection_edges.append({
        "id":f"p{j}",
        "from":a,
        "to":b,
        "weight":int(d["weight"]),
        "edge_betweenness":edge_betweenness.get((a,b),edge_betweenness.get((b,a),0)),
    })

print("Community method:", community_method)
print("Communities:", len(communities))
print("Community size range: 1 to", MAX_COMMUNITY_SIZE)


In [ ]:

proj_html = f"""<!doctype html>
<html>
<head>
<title>Projected university network</title>
{common_head}
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
</head>
<body>

<header>
<h1>University Projection by Shared Students</h1>
<div class="sub">
Visible labels use raw University names. Community colors are optional; community filtering is based on community size.
</div>
</header>

<div class="controls">

<div class="control">
<label>Minimum shared students</label>
<input id="minWeight" type="range" min="1" max="{max(e["weight"] for e in projection_edges)}" value="1">
<span class="small" id="minWeightLabel">1</span>
</div>

<div class="control">
<label>Minimum visible degree</label>
<input id="minDegree" type="number" min="1" max="50" value="1">
</div>

<div class="control">
<label>Community size min</label>
<input id="communityMin" type="range" min="1" max="{MAX_COMMUNITY_SIZE}" value="1">
<span class="small" id="communityMinLabel">1</span>
</div>

<div class="control">
<label>Community size max</label>
<input id="communityMax" type="range" min="1" max="{MAX_COMMUNITY_SIZE}" value="{MAX_COMMUNITY_SIZE}">
<span class="small" id="communityMaxLabel">{MAX_COMMUNITY_SIZE}</span>
</div>

<div class="control">
<label>University</label>
<select id="uniSearch">
<option value="">None</option>
{option_tags(UNIVERSITY_OPTIONS, university_display)}
</select>
</div>

<div class="control">
<label>Node size by</label>
<select id="sizeBy">
<option value="students">Student count</option>
<option value="degree">Full-network degree</option>
<option value="weighted_degree">Weighted degree</option>
<option value="degree_centrality">Degree centrality</option>
<option value="betweenness">Betweenness</option>
<option value="closeness">Closeness</option>
<option value="pagerank">PageRank</option>
</select>
</div>

<div class="control">
<label>Edge width by</label>
<select id="edgeBy">
<option value="weight">Shared students</option>
<option value="edge_betweenness">Edge betweenness</option>
</select>
</div>

<div class="control">
<label>Community colors</label>
<div class="inline-check">
<input id="colorCommunities" type="checkbox">
<span>Color nodes</span>
</div>
</div>

<div class="control">
<label>Physics</label>
<div>
<button type="button" id="stopBtn">Stop</button>
<button type="button" id="slowBtn">Slow</button>
<button type="button" id="normalBtn" class="active-btn">Normal</button>
<button type="button" id="fastBtn">Fast</button>
</div>
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="net"></div>

<script>
const N={json_compact(projection_nodes)};
const E={json_compact(projection_edges)};
const container=document.getElementById("net");
let network=null;
let physicsMode="normal";

function makeTooltip(lines){{
    const div=document.createElement("div");

    lines.forEach(line=>{{
        const row=document.createElement("div");

        if(line.bold){{
            const strong=document.createElement("strong");
            strong.textContent=line.text;
            row.appendChild(strong);
        }} else {{
            row.textContent=line.text;
        }}

        div.appendChild(row);
    }});

    return div;
}}

function physicsOptions(mode){{
    if(mode==="stop") return {{enabled:false}};

    const speed={{
        slow:{{timestep:.22,maxVelocity:18}},
        normal:{{timestep:.50,maxVelocity:38}},
        fast:{{timestep:.85,maxVelocity:70}}
    }}[mode];

    return {{
        enabled:true,
        solver:"forceAtlas2Based",
        timestep:speed.timestep,
        maxVelocity:speed.maxVelocity,
        forceAtlas2Based:{{gravitationalConstant:-55,springLength:100}},
        stabilization:{{enabled:false}}
    }};
}}

function setPhysics(mode){{
    physicsMode=mode;

    ["stopBtn","slowBtn","normalBtn","fastBtn"]
        .forEach(id=>document.getElementById(id).classList.remove("active-btn"));

    document.getElementById(
        mode==="stop"?"stopBtn":
        mode==="slow"?"slowBtn":
        mode==="fast"?"fastBtn":"normalBtn"
    ).classList.add("active-btn");

    if(network) network.setOptions({{physics:physicsOptions(mode)}});
}}

function scale(v,min,max,a,b){{
    if(max<=min) return (a+b)/2;
    return a+(v-min)*(b-a)/(max-min);
}}

function communityColor(cid){{
    if(!cid) return "#4c78a8";
    const hue=(cid*137.508)%360;
    return `hsl(${{hue}},62%,46%)`;
}}

function thresholdGraph(minW,minD,cmin,cmax){{
    const allowedNodes=new Set(
        N.filter(n=>n.community_size>=cmin && n.community_size<=cmax)
         .map(n=>n.id)
    );

    let edges=E.filter(
        e=>e.weight>=minW &&
        allowedNodes.has(e.from) &&
        allowedNodes.has(e.to)
    );

    let active=new Set();
    edges.forEach(e=>{{active.add(e.from);active.add(e.to);}});

    let changed=true;

    while(changed){{
        changed=false;
        const degree={{}};

        edges.forEach(e=>{{
            if(active.has(e.from)&&active.has(e.to)){{
                degree[e.from]=(degree[e.from]||0)+1;
                degree[e.to]=(degree[e.to]||0)+1;
            }}
        }});

        const remove=[...active].filter(id=>(degree[id]||0)<minD);

        if(remove.length){{
            changed=true;
            remove.forEach(id=>active.delete(id));
            edges=edges.filter(e=>active.has(e.from)&&active.has(e.to));
        }}
    }}

    const incident=new Set();
    edges.forEach(e=>{{incident.add(e.from);incident.add(e.to);}});
    active=new Set([...active].filter(id=>incident.has(id)));

    return {{active,edges}};
}}

function render(){{
    const mw=+minWeight.value;
    const md=Math.max(1,parseInt(minDegree.value||"1",10));

    let cmin=+communityMin.value;
    let cmax=+communityMax.value;
    if(cmin>cmax) [cmin,cmax]=[cmax,cmin];

    const sb=sizeBy.value;
    const eb=edgeBy.value;
    const selected=uniSearch.value;
    const useCommunityColors=colorCommunities.checked;

    const tg=thresholdGraph(mw,md,cmin,cmax);
    const nodes=N.filter(n=>tg.active.has(n.id));

    const vals=nodes.map(n=>+n[sb]||0);
    const mn=Math.min(...vals,0);
    const mx=Math.max(...vals,1);

    const eVals=tg.edges.map(e=>+e[eb]||0);
    const emn=Math.min(...eVals,0);
    const emx=Math.max(...eVals,1);

    const visualNodes=nodes.map(n=>{{
        const location=[n.city,n.state]
            .filter(x=>x && x.trim())
            .join(", ");

        return {{
            ...n,
            size:scale(+n[sb]||0,mn,mx,8,35),
            color:selected===n.id ? "#e15759" :
                  (useCommunityColors ? communityColor(n.community) : "#4c78a8"),
            title:makeTooltip([
                {{text:n.label,bold:true}},
                ...(location ? [{{text:location}}] : []),
                ...(n.country ? [{{text:n.country}}] : []),
                {{text:`Students: ${{n.students}}`}},
                {{text:`Community: ${{n.community}} (size ${{n.community_size}})`}},
                {{text:`Full-network degree: ${{n.degree}}`}},
                {{text:`Weighted degree: ${{n.weighted_degree}}`}},
                {{text:`Betweenness: ${{n.betweenness.toFixed(4)}}`}}
            ])
        }};
    }});

    const visualEdges=tg.edges.map(e=>{{
        const fromNode=N.find(n=>n.id===e.from);
        const toNode=N.find(n=>n.id===e.to);

        return {{
            ...e,
            width:scale(+e[eb]||0,emn,emx,.6,8),
            color:{{color:"#aab2bd"}},
            title:makeTooltip([
                {{
                    text:`${{fromNode?.label || e.from}} ↔ ${{toNode?.label || e.to}}`,
                    bold:true
                }},
                {{text:`Shared students: ${{e.weight}}`}},
                {{text:`Edge betweenness: ${{e.edge_betweenness.toFixed(5)}}`}}
            ])
        }};
    }});

    if(network) network.destroy();

    network=new vis.Network(
        container,
        {{nodes:new vis.DataSet(visualNodes),edges:new vis.DataSet(visualEdges)}},
        {{
            physics:physicsOptions(physicsMode),
            interaction:{{hover:true,navigationButtons:true}},
            nodes:{{shape:"dot",font:{{size:11}}}}
        }}
    );

    if(selected && tg.active.has(selected)){{
        network.selectNodes([selected]);
        network.focus(selected,{{scale:1.4,animation:true}});
    }}

    const searchNote=selected && !tg.active.has(selected)
        ? " · selected university filtered out"
        : "";

    stat.textContent=
        `${{visualNodes.length}} universities · ${{visualEdges.length}} edges · `+
        `community sizes ${{cmin}}–${{cmax}}${{searchNote}}`;
}}

minWeight.oninput=()=>{{
    minWeightLabel.textContent=minWeight.value;
    render();
}};

communityMin.oninput=()=>{{
    communityMinLabel.textContent=communityMin.value;
    render();
}};

communityMax.oninput=()=>{{
    communityMaxLabel.textContent=communityMax.value;
    render();
}};

minDegree.onchange=render;
uniSearch.onchange=render;
sizeBy.onchange=render;
edgeBy.onchange=render;
colorCommunities.onchange=render;

stopBtn.onclick=()=>setPhysics("stop");
slowBtn.onclick=()=>setPhysics("slow");
normalBtn.onclick=()=>setPhysics("normal");
fastBtn.onclick=()=>setPhysics("fast");

reset.onclick=()=>{{
    minWeight.value=1;
    minWeightLabel.textContent="1";
    minDegree.value=1;
    communityMin.value=1;
    communityMinLabel.textContent="1";
    communityMax.value="{MAX_COMMUNITY_SIZE}";
    communityMaxLabel.textContent="{MAX_COMMUNITY_SIZE}";
    uniSearch.value="";
    sizeBy.value="students";
    edgeBy.value="weight";
    colorCommunities.checked=false;
    physicsMode="normal";
    render();
    setPhysics("normal");
}};

render();
</script>
</body>
</html>
"""

proj_path = OUTPUT_DIR / "02_university_projection.html"
proj_path.write_text(proj_html, encoding="utf-8")
print("Saved:", proj_path.resolve())


## 3. Spatial network with community-size filtering

In [ ]:

geo = {}

for u,g in df.groupby("_uni"):
    gg=g.dropna(subset=["Latitude","Longitude"]).copy()

    if len(gg)==0:
        continue

    lat=pd.to_numeric(gg["Latitude"],errors="coerce").median()
    lon=pd.to_numeric(gg["Longitude"],errors="coerce").median()

    if pd.isna(lat) or pd.isna(lon):
        continue

    cid=community_of.get(u,0)

    geo[u]={
        "lat":float(lat),
        "lon":float(lon),
        "country":uni_meta[u]["country"],
        "city":uni_meta[u]["city"],
        "students":len(uni_students[u]),
        "community":cid,
        "community_size":community_sizes.get(cid,1),
    }

spatial_nodes=[
    {
        "id":u,
        "label":university_display[u],
        "clean_label":u,
        **meta
    }
    for u,meta in geo.items()
]

spatial_edges=[
    e for e in projection_edges
    if e["from"] in geo and e["to"] in geo
]

print("Geolocated universities:",len(spatial_nodes))


In [ ]:

spatial_html = f"""<!doctype html>
<html>
<head>
<title>Spatial university network</title>
{common_head}
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
#legend{{position:absolute;z-index:1000;right:12px;bottom:12px;max-height:260px;overflow:auto;background:rgba(255,255,255,.94);border:1px solid #ddd;border-radius:8px;padding:8px 10px;font-size:11px}}
.legend-row{{display:flex;align-items:center;gap:6px;margin:3px 0}}
.legend-dot{{width:11px;height:11px;border-radius:50%;display:inline-block}}
</style>
</head>
<body>

<header>
<h1>Spatial University Network — Communities</h1>
<div class="sub">
Visible labels use raw University names. Communities are filtered by size interval rather than ID.
</div>
</header>

<div class="controls">

<div class="control">
<label>Map scope</label>
<select id="scope">
<option value="world">World</option>
<option value="us">United States</option>
</select>
</div>

<div class="control">
<label>Minimum shared students</label>
<input id="minWeight" type="range" min="1" max="{max(e["weight"] for e in spatial_edges)}" value="1">
<span class="small" id="minWeightLabel">1</span>
</div>

<div class="control">
<label>Minimum visible degree</label>
<input id="minDegree" type="number" min="1" max="50" value="1">
</div>

<div class="control">
<label>Community size min</label>
<input id="communityMin" type="range" min="1" max="{MAX_COMMUNITY_SIZE}" value="1">
<span class="small" id="communityMinLabel">1</span>
</div>

<div class="control">
<label>Community size max</label>
<input id="communityMax" type="range" min="1" max="{MAX_COMMUNITY_SIZE}" value="{MAX_COMMUNITY_SIZE}">
<span class="small" id="communityMaxLabel">{MAX_COMMUNITY_SIZE}</span>
</div>

<div class="control">
<label>University</label>
<select id="uniSearch">
<option value="">None</option>
{option_tags(sorted(geo.keys(),key=lambda u:university_display[u].lower()),university_display)}
</select>
</div>

<div class="control">
<label>Community colors</label>
<div class="inline-check">
<input id="colorCommunities" type="checkbox" checked>
<span>Color nodes</span>
</div>
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div style="position:relative">
<div id="map"></div>
<div id="legend"></div>
</div>

<script>
const N={json_compact(spatial_nodes)};
const E={json_compact(spatial_edges)};
const byId=Object.fromEntries(N.map(n=>[n.id,n]));

const map=L.map("map",{{worldCopyJump:true}}).setView([30,10],2);

L.tileLayer(
    "https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png",
    {{maxZoom:18,attribution:"© OpenStreetMap contributors"}}
).addTo(map);

let layer=L.layerGroup().addTo(map);

function isUS(country){{
    return [
        "united states","united states of america",
        "usa","u.s.","us","u.s.a."
    ].includes((country||"").trim().toLowerCase());
}}

function communityColor(cid){{
    if(!cid) return "#8c8c8c";
    const hue=(cid*137.508)%360;
    return `hsl(${{hue}},62%,46%)`;
}}

function thresholdGraph(baseNodes,minW,minD){{
    const allowed=new Set(baseNodes.map(n=>n.id));

    let edges=E.filter(
        e=>e.weight>=minW &&
        allowed.has(e.from) &&
        allowed.has(e.to)
    );

    let active=new Set();
    edges.forEach(e=>{{active.add(e.from);active.add(e.to);}});

    let changed=true;

    while(changed){{
        changed=false;
        const degree={{}};

        edges.forEach(e=>{{
            if(active.has(e.from)&&active.has(e.to)){{
                degree[e.from]=(degree[e.from]||0)+1;
                degree[e.to]=(degree[e.to]||0)+1;
            }}
        }});

        const remove=[...active].filter(id=>(degree[id]||0)<minD);

        if(remove.length){{
            changed=true;
            remove.forEach(id=>active.delete(id));
            edges=edges.filter(e=>active.has(e.from)&&active.has(e.to));
        }}
    }}

    const incident=new Set();
    edges.forEach(e=>{{incident.add(e.from);incident.add(e.to);}});
    active=new Set([...active].filter(id=>incident.has(id)));

    return {{active,edges}};
}}

function render(){{
    layer.clearLayers();

    const sc=scope.value;
    const mw=+minWeight.value;
    const md=Math.max(1,parseInt(minDegree.value||"1",10));

    let cmin=+communityMin.value;
    let cmax=+communityMax.value;
    if(cmin>cmax) [cmin,cmax]=[cmax,cmin];

    const selected=uniSearch.value;
    const useCommunityColors=colorCommunities.checked;

    const baseNodes=N.filter(n=>
        (sc==="world" || isUS(n.country)) &&
        n.community_size>=cmin &&
        n.community_size<=cmax
    );

    const tg=thresholdGraph(baseNodes,mw,md);
    const visibleNodes=baseNodes.filter(n=>tg.active.has(n.id));

    tg.edges.forEach(e=>{{
        const a=byId[e.from];
        const b=byId[e.to];

        L.polyline(
            [[a.lat,a.lon],[b.lat,b.lon]],
            {{
                weight:Math.max(.5,Math.min(7,.6+e.weight*.45)),
                opacity:.28,
                color:"#7b8494"
            }}
        )
        .bindTooltip(
            `${{a.label}} ↔ ${{b.label}}<br>Shared students: ${{e.weight}}`
        )
        .addTo(layer);
    }});

    visibleNodes.forEach(n=>{{
        const hit=selected===n.id;
        const color=useCommunityColors ? communityColor(n.community) : "#4c78a8";

        const marker=L.circleMarker(
            [n.lat,n.lon],
            {{
                radius:hit?11:Math.max(4,Math.min(13,3+Math.sqrt(n.students))),
                color,
                fillColor:color,
                fillOpacity:.78,
                weight:hit?4:1.5
            }}
        )
        .bindTooltip(
            `<b>${{n.label}}</b>`+
            `<br>Internal key: ${{n.clean_label}}`+
            `<br>${{n.city||""}} ${{n.country||""}}`+
            `<br>Students: ${{n.students}}`+
            `<br>Community: ${{n.community}} (size ${{n.community_size}})`
        )
        .addTo(layer);

        if(hit){{
            marker.openTooltip();
            map.panTo([n.lat,n.lon]);
        }}
    }});

    const commCounts={{}};
    visibleNodes.forEach(n=>{{
        if(!commCounts[n.community]) commCounts[n.community]=0;
        commCounts[n.community]+=1;
    }});

    const visibleC=Object.keys(commCounts)
        .map(Number)
        .sort((a,b)=>commCounts[b]-commCounts[a]);

    legend.innerHTML = useCommunityColors
        ? "<b>Visible communities</b>"+
          visibleC.slice(0,15).map(cid=>
              `<div class="legend-row">`+
              `<span class="legend-dot" style="background:${{communityColor(cid)}}"></span>`+
              `<span>Community ${{cid}} · ${{commCounts[cid]}} universities</span>`+
              `</div>`
          ).join("")
        : "<b>Community colors off</b><div class='small'>Nodes use a single color; community-size filtering remains active.</div>";

    if(!selected){{
        if(sc==="us") map.setView([39,-98],4);
        else map.setView([25,10],2);
    }}

    const searchNote=selected && !tg.active.has(selected)
        ? " · selected university filtered out"
        : "";

    stat.textContent=
        `${{visibleNodes.length}} universities · ${{tg.edges.length}} edges · `+
        `community sizes ${{cmin}}–${{cmax}}${{searchNote}}`;
}}

scope.onchange=render;

minWeight.oninput=()=>{{
    minWeightLabel.textContent=minWeight.value;
    render();
}};

communityMin.oninput=()=>{{
    communityMinLabel.textContent=communityMin.value;
    render();
}};

communityMax.oninput=()=>{{
    communityMaxLabel.textContent=communityMax.value;
    render();
}};

minDegree.onchange=render;
uniSearch.onchange=render;
colorCommunities.onchange=render;

reset.onclick=()=>{{
    scope.value="world";
    minWeight.value=1;
    minWeightLabel.textContent="1";
    minDegree.value=1;
    communityMin.value=1;
    communityMinLabel.textContent="1";
    communityMax.value="{MAX_COMMUNITY_SIZE}";
    communityMaxLabel.textContent="{MAX_COMMUNITY_SIZE}";
    uniSearch.value="";
    colorCommunities.checked=true;
    render();
}};

render();
</script>
</body>
</html>
"""

spatial_path=OUTPUT_DIR/"03_spatial_network.html"
spatial_path.write_text(spatial_html,encoding="utf-8")
print("Saved:",spatial_path.resolve())


## 4. Transition events with separate source and target levels

In [ ]:

# One institution-year record per student and university.
# Multiple normalized levels can coexist for a student-university-year observation.

record_rows=[]

for (sid,uni,year),g in df.dropna(subset=["_year_num"]).groupby(
    ["_student","_uni","_year_num"]
):
    levels=sorted(set(g["_level_cat"].tolist()))

    record_rows.append({
        "student":sid,
        "university":uni,
        "display":university_display[uni],
        "year":int(year),
        "levels":levels,
        "country_class":university_country_class.get(uni,"Other"),
    })

record_df=pd.DataFrame(record_rows)

transition_events=[]

for sid,g in record_df.groupby("student"):
    by_year={
        int(year):gg.to_dict("records")
        for year,gg in g.groupby("year")
    }

    years=sorted(by_year)

    for y1,y2 in zip(years,years[1:]):
        for src in by_year[y1]:
            for dst in by_year[y2]:
                if src["university"]==dst["university"]:
                    continue

                transition_events.append({
                    "student":sid,
                    "source":src["university"],
                    "source_display":src["display"],
                    "target":dst["university"],
                    "target_display":dst["display"],
                    "source_year":y1,
                    "target_year":y2,
                    "source_levels":src["levels"],
                    "target_levels":dst["levels"],
                    "source_country":src["country_class"],
                    "target_country":dst["country_class"],
                })

print("Transition events:",len(transition_events))


## 5. General three-stage pathways across all countries

In [ ]:

# Three consecutive observed education years across ALL countries.
# Each sliding three-year window contributes combinations across Stage 1 -> Stage 2 -> Stage 3.

multistage_paths=[]

for sid,g in record_df.groupby("student"):
    by_year={
        int(year):gg.to_dict("records")
        for year,gg in g.groupby("year")
    }

    years=sorted(by_year)

    for y1,y2,y3 in zip(years,years[1:],years[2:]):
        for s1 in by_year[y1]:
            for s2 in by_year[y2]:
                for s3 in by_year[y3]:

                    if s1["university"]==s2["university"]:
                        continue
                    if s2["university"]==s3["university"]:
                        continue

                    multistage_paths.append({
                        "student":sid,

                        "stage1":s1["university"],
                        "stage1_display":s1["display"],
                        "stage1_year":y1,
                        "stage1_levels":s1["levels"],
                        "stage1_country":s1["country_class"],

                        "stage2":s2["university"],
                        "stage2_display":s2["display"],
                        "stage2_year":y2,
                        "stage2_levels":s2["levels"],
                        "stage2_country":s2["country_class"],

                        "stage3":s3["university"],
                        "stage3_display":s3["display"],
                        "stage3_year":y3,
                        "stage3_levels":s3["levels"],
                        "stage3_country":s3["country_class"],
                    })

print("Three-stage path records:",len(multistage_paths))

country_patterns=Counter(
    (
        p["stage1_country"],
        p["stage2_country"],
        p["stage3_country"],
    )
    for p in multistage_paths
)

print("\nMost common country sequences:")
for pattern,count in country_patterns.most_common(15):
    print(pattern,count)


## 6. Sankey builders with independent level filters

In [ ]:

def build_transition_sankey_html(
    events,
    mode,
    title,
    subtitle,
    output_path,
    default_top_n=40,
):
    mode_condition_js={
        "china_us":'e.source_country==="China" && e.target_country==="US"',
        "us_us":'e.source_country==="US" && e.target_country==="US"',
        "other":'e.source_country==="Other" || e.target_country==="Other"',
    }[mode]

    payload=json_compact(events)

    html_text=f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(title)}</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a;background:#fff}}
header{{padding:14px 18px 8px;border-bottom:1px solid #e4e7ec}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #e4e7ec;align-items:end}}
.control{{display:flex;flex-direction:column;gap:4px}}
label{{font-size:11px;font-weight:650;color:#475467}}
input,select,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:#fff}}
select[multiple]{{min-width:150px;height:38px;min-height:38px;padding:3px;vertical-align:bottom}}
button{{cursor:pointer}}#chart{{height:780px}}.stat{{margin-left:auto;color:#667085;font-size:12px;padding:8px}}
.small{{font-size:11px;color:#667085}}
</style>
</head>
<body>

<header>
<h1>{html.escape(title)}</h1>
<div class="sub">{html.escape(subtitle)}</div>
</header>

<div class="controls">

<div class="control">
<label>Year from</label>
<input id="yearMin" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MIN}">
</div>

<div class="control">
<label>Year to</label>
<input id="yearMax" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MAX}">
</div>

{multi_level_select("sourceLevelFilter","Source / start level(s)",size=6)}

{multi_level_select("targetLevelFilter","Target / end level(s)",size=6)}

<div class="control">
<label>Top N flows</label>
<input id="topN" type="number" min="5" max="500" value="{default_top_n}">
</div>

<div class="control">
<label>Minimum students per flow</label>
<input id="minStudents" type="number" min="1" max="100" value="1">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="chart"></div>

<script>
const EVENTS={payload};

function selectedValues(el){{
    return [...el.selectedOptions].map(o=>o.value);
}}

function anyMatch(recordLevels,selected){{
    return selected.length===0 || selected.some(level=>(recordLevels||[]).includes(level));
}}

function render(){{
    let y1=parseInt(yearMin.value,10);
    let y2=parseInt(yearMax.value,10);
    if(y1>y2) [y1,y2]=[y2,y1];

    const sourceLevels=selectedValues(sourceLevelFilter);
    const targetLevels=selectedValues(targetLevelFilter);
    const minS=Math.max(1,parseInt(minStudents.value||"1",10));
    const top=Math.max(5,parseInt(topN.value||"{default_top_n}",10));

    const filtered=EVENTS.filter(e=>
        ({mode_condition_js}) &&
        e.source_year>=y1 &&
        e.target_year<=y2 &&
        anyMatch(e.source_levels,sourceLevels) &&
        anyMatch(e.target_levels,targetLevels)
    );

    const pairStudents=new Map();
    const pairLabels=new Map();

    filtered.forEach(e=>{{
        const key=e.source+"|||"+e.target;

        if(!pairStudents.has(key)){{
            pairStudents.set(key,new Set());
            pairLabels.set(key,[e.source_display,e.target_display]);
        }}

        pairStudents.get(key).add(e.student);
    }});

    let flows=[...pairStudents.entries()].map(([key,set])=>{{
        const [source,target]=key.split("|||");
        const labels=pairLabels.get(key);
        return {{
            source,
            target,
            source_display:labels[0],
            target_display:labels[1],
            students:set.size
        }};
    }});

    flows=flows
        .filter(d=>d.students>=minS)
        .sort((a,b)=>
            b.students-a.students ||
            a.source_display.localeCompare(b.source_display) ||
            a.target_display.localeCompare(b.target_display)
        )
        .slice(0,top);

    const nodeKeys=[];
    const nodeLabels={{}};

    flows.forEach(d=>{{
        const s="S:"+d.source;
        const t="T:"+d.target;

        if(!nodeKeys.includes(s)) nodeKeys.push(s);
        if(!nodeKeys.includes(t)) nodeKeys.push(t);

        nodeLabels[s]=d.source_display;
        nodeLabels[t]=d.target_display;
    }});

    const idx=Object.fromEntries(nodeKeys.map((x,i)=>[x,i]));

    const trace={{
        type:"sankey",
        arrangement:"snap",
        node:{{
            pad:14,
            thickness:16,
            label:nodeKeys.map(k=>nodeLabels[k]),
            line:{{color:"rgba(40,40,40,.35)",width:.5}}
        }},
        link:{{
            source:flows.map(d=>idx["S:"+d.source]),
            target:flows.map(d=>idx["T:"+d.target]),
            value:flows.map(d=>d.students),
            customdata:flows.map(
                d=>`${{d.source_display}} → ${{d.target_display}}: ${{d.students}} students`
            ),
            hovertemplate:"%{{customdata}}<extra></extra>"
        }}
    }};

    Plotly.react(
        "chart",
        [trace],
        {{margin:{{l:25,r:25,t:20,b:25}},font:{{size:11}},autosize:true}},
        {{responsive:true,displaylogo:false}}
    );

    stat.textContent=
        `${{flows.length}} flows · ${{nodeKeys.length}} stage-specific nodes · years ${{y1}}–${{y2}}`;
}}

["yearMin","yearMax","sourceLevelFilter","targetLevelFilter","topN","minStudents"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    yearMin.value="{YEAR_MIN}";
    yearMax.value="{YEAR_MAX}";
    [...sourceLevelFilter.options].forEach(o=>o.selected=false);
    [...targetLevelFilter.options].forEach(o=>o.selected=false);
    topN.value="{default_top_n}";
    minStudents.value="1";
    render();
}};

render();
</script>
</body>
</html>
"""

    output_path=Path(output_path)
    output_path.write_text(html_text,encoding="utf-8")
    return output_path


def build_multistage_sankey_html(
    paths,
    title,
    subtitle,
    output_path,
    default_top_n=40,
):
    payload=json_compact(paths)

    html_text=f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(title)}</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a;background:#fff}}
header{{padding:14px 18px 8px;border-bottom:1px solid #e4e7ec}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #e4e7ec;align-items:end}}
.control{{display:flex;flex-direction:column;gap:4px}}
label{{font-size:11px;font-weight:650;color:#475467}}
input,select,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:#fff}}
select[multiple]{{min-width:150px;height:38px;min-height:38px;padding:3px;vertical-align:bottom}}
button{{cursor:pointer}}#chart{{height:840px}}.stat{{margin-left:auto;color:#667085;font-size:12px;padding:8px}}
.small{{font-size:11px;color:#667085}}
</style>
</head>
<body>

<header>
<h1>{html.escape(title)}</h1>
<div class="sub">{html.escape(subtitle)}</div>
</header>

<div class="controls">

<div class="control">
<label>Year from</label>
<input id="yearMin" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MIN}">
</div>

<div class="control">
<label>Year to</label>
<input id="yearMax" type="number" min="{YEAR_MIN}" max="{YEAR_MAX}" value="{YEAR_MAX}">
</div>

{multi_level_select("stage1LevelFilter","Stage 1 / start level(s)",size=6)}
{multi_level_select("stage2LevelFilter","Stage 2 / middle level(s)",size=6)}
{multi_level_select("stage3LevelFilter","Stage 3 / end level(s)",size=6)}

<div class="control">
<label>Top N links per stage</label>
<input id="topN" type="number" min="5" max="400" value="{default_top_n}">
</div>

<div class="control">
<label>Minimum students per link</label>
<input id="minStudents" type="number" min="1" max="100" value="1">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="chart"></div>

<script>
const PATHS={payload};

function selectedValues(el){{
    return [...el.selectedOptions].map(o=>o.value);
}}

function anyMatch(recordLevels,selected){{
    return selected.length===0 || selected.some(level=>(recordLevels||[]).includes(level));
}}

function aggregateLinks(paths,sourceKey,targetKey,sourceDisplayKey,targetDisplayKey,sourcePrefix,targetPrefix){{
    const m=new Map();
    const labels=new Map();

    paths.forEach(p=>{{
        const s=sourcePrefix+p[sourceKey];
        const t=targetPrefix+p[targetKey];
        const key=s+"|||"+t;

        if(!m.has(key)){{
            m.set(key,new Set());
            labels.set(key,[p[sourceDisplayKey],p[targetDisplayKey]]);
        }}

        m.get(key).add(p.student);
    }});

    return [...m.entries()].map(([key,set])=>{{
        const [source,target]=key.split("|||");
        const lbl=labels.get(key);
        return {{
            source,
            target,
            source_display:lbl[0],
            target_display:lbl[1],
            students:set.size
        }};
    }});
}}

function render(){{
    let y1=parseInt(yearMin.value,10);
    let y2=parseInt(yearMax.value,10);
    if(y1>y2) [y1,y2]=[y2,y1];

    const l1=selectedValues(stage1LevelFilter);
    const l2=selectedValues(stage2LevelFilter);
    const l3=selectedValues(stage3LevelFilter);

    const minS=Math.max(1,parseInt(minStudents.value||"1",10));
    const top=Math.max(5,parseInt(topN.value||"{default_top_n}",10));

    const filtered=PATHS.filter(p=>
        p.stage1_year>=y1 &&
        p.stage3_year<=y2 &&
        anyMatch(p.stage1_levels,l1) &&
        anyMatch(p.stage2_levels,l2) &&
        anyMatch(p.stage3_levels,l3)
    );

    let links1=aggregateLinks(
        filtered,
        "stage1","stage2",
        "stage1_display","stage2_display",
        "S1:","S2:"
    )
    .filter(d=>d.students>=minS)
    .sort((a,b)=>b.students-a.students)
    .slice(0,top);

    let links2=aggregateLinks(
        filtered,
        "stage2","stage3",
        "stage2_display","stage3_display",
        "S2:","S3:"
    )
    .filter(d=>d.students>=minS)
    .sort((a,b)=>b.students-a.students)
    .slice(0,top);

    const links=[...links1,...links2];
    const nodeKeys=[];
    const nodeLabels={{}};

    links.forEach(d=>{{
        if(!nodeKeys.includes(d.source)) nodeKeys.push(d.source);
        if(!nodeKeys.includes(d.target)) nodeKeys.push(d.target);

        nodeLabels[d.source]=d.source_display;
        nodeLabels[d.target]=d.target_display;
    }});

    const idx=Object.fromEntries(nodeKeys.map((x,i)=>[x,i]));

    const trace={{
        type:"sankey",
        arrangement:"snap",
        node:{{
            pad:14,
            thickness:16,
            label:nodeKeys.map(k=>nodeLabels[k]),
            line:{{color:"rgba(40,40,40,.35)",width:.5}}
        }},
        link:{{
            source:links.map(d=>idx[d.source]),
            target:links.map(d=>idx[d.target]),
            value:links.map(d=>d.students),
            customdata:links.map(
                d=>`${{d.source_display}} → ${{d.target_display}}: ${{d.students}} students`
            ),
            hovertemplate:"%{{customdata}}<extra></extra>"
        }}
    }};

    Plotly.react(
        "chart",
        [trace],
        {{margin:{{l:25,r:25,t:20,b:25}},font:{{size:10}},autosize:true}},
        {{responsive:true,displaylogo:false}}
    );

    stat.textContent=
        `${{filtered.length}} qualifying three-stage path records · `+
        `${{links.length}} displayed links · years ${{y1}}–${{y2}}`;
}}

[
    "yearMin","yearMax",
    "stage1LevelFilter","stage2LevelFilter","stage3LevelFilter",
    "topN","minStudents"
]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    yearMin.value="{YEAR_MIN}";
    yearMax.value="{YEAR_MAX}";

    [...stage1LevelFilter.options].forEach(o=>o.selected=false);
    [...stage2LevelFilter.options].forEach(o=>o.selected=false);
    [...stage3LevelFilter.options].forEach(o=>o.selected=false);

    topN.value="{default_top_n}";
    minStudents.value="1";
    render();
}};

render();
</script>
</body>
</html>
"""

    output_path=Path(output_path)
    output_path.write_text(html_text,encoding="utf-8")
    return output_path


In [ ]:

china_us_sankey=build_transition_sankey_html(
    transition_events,
    mode="china_us",
    title="Student Flows: China → United States",
    subtitle=(
        "Source/start and target/end levels are filtered independently. "
        "Flow width = distinct students."
    ),
    output_path=OUTPUT_DIR/"04_sankey_china_to_us.html",
    default_top_n=40,
)

us_us_sankey=build_transition_sankey_html(
    transition_events,
    mode="us_us",
    title="Student Flows Among U.S. Universities",
    subtitle=(
        "Source/start and target/end levels are filtered independently. "
        "Flow width = distinct students."
    ),
    output_path=OUTPUT_DIR/"05_sankey_us_to_us.html",
    default_top_n=40,
)

other_sankey=build_transition_sankey_html(
    transition_events,
    mode="other",
    title="Student Flows Involving Universities Outside China / U.S.",
    subtitle=(
        "Includes Other↔Other, China↔Other, and U.S.↔Other transitions, "
        "with separate source and target level filters."
    ),
    output_path=OUTPUT_DIR/"06_sankey_other_countries.html",
    default_top_n=50,
)

multistage_sankey=build_multistage_sankey_html(
    multistage_paths,
    title="Multi-stage Student Pathways Across All Countries",
    subtitle=(
        "Three consecutive observed education stages. "
        "China, U.S., and other-country institutions are all included. "
        "Stage 1, Stage 2, and Stage 3 levels are filtered independently."
    ),
    output_path=OUTPUT_DIR/"07_sankey_multistage_all_countries.html",
    default_top_n=40,
)

for p in [
    china_us_sankey,
    us_us_sankey,
    other_sankey,
    multistage_sankey,
]:
    print("Saved:",p.resolve())


## 7. Export supporting tables

In [ ]:

projection_node_df=pd.DataFrame(projection_nodes)
projection_edge_df=pd.DataFrame(projection_edges)
transition_event_df=pd.DataFrame(transition_events)
multistage_df=pd.DataFrame(multistage_paths)

projection_node_df.to_csv(
    OUTPUT_DIR/"university_node_metrics.csv",
    index=False
)

projection_edge_df.to_csv(
    OUTPUT_DIR/"university_edge_metrics.csv",
    index=False
)

transition_event_df.to_csv(
    OUTPUT_DIR/"student_transition_events.csv",
    index=False
)

multistage_df.to_csv(
    OUTPUT_DIR/"multistage_paths_all_countries.csv",
    index=False
)

display_map_df.to_csv(
    OUTPUT_DIR/"university_display_mapping.csv",
    index=False
)

print("Output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -",p)


## 8. Preview interactive outputs

In [ ]:
display(IFrame(src=str(bip_path), width="100%", height=820))

In [ ]:
display(IFrame(src=str(proj_path), width="100%", height=820))

In [ ]:
display(IFrame(src=str(spatial_path), width="100%", height=820))

In [ ]:
display(IFrame(src=str(china_us_sankey), width="100%", height=820))

In [ ]:
display(IFrame(src=str(us_us_sankey), width="100%", height=820))

In [ ]:
display(IFrame(src=str(other_sankey), width="100%", height=820))

In [ ]:
display(IFrame(src=str(multistage_sankey), width="100%", height=820))